# RARE EVENT OPTIMIZATION FOR BANK EWS

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import precision_recall_curve, average_precision_score
from xgboost import XGBClassifier
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')


print("="*70)
print("RARE EVENT OPTIMIZATION - Bank Early Warning System")
print("Improving performance when stress prevalence < 10%")
print("="*70)

# Load data
df = pd.read_csv(r"D:\Bank_EWS_Project\final_data\bank_ml_dataset.csv")

# Create lagged features
df_lagged = df.copy()
df_lagged = df_lagged.sort_values(['bank_name', 'year'])

all_features = ['crar_total', 'npa_ratio', 'total_provisions', 'net_profit', 
                'interest_income', 'interest_expense', 'operating_expense',
                'credit_growth', 'repo_rate', 'inflation']

for col in all_features:
    df_lagged[f'{col}_lag1'] = df_lagged.groupby('bank_name')[col].shift(1)

df_lagged = df_lagged.dropna(subset=[f'{col}_lag1' for col in all_features])
lagged_feature_cols = [f'{col}_lag1' for col in all_features]

# Split: Train on ALL pre-2023 data, test on 2023-2025 (rare event period)
train_data = df_lagged[df_lagged['year'] < 2023]
test_data = df_lagged[df_lagged['year'] >= 2023]

X_train = train_data[lagged_feature_cols]
y_train = train_data['stress_label']
X_test = test_data[lagged_feature_cols]
y_test = test_data['stress_label']

print(f"\n📊 Data Split:")
print(f"   Training (2015-2022): {len(X_train)} samples, {y_train.sum()} stressed ({y_train.mean()*100:.1f}%)")
print(f"   Test (2023-2025): {len(X_test)} samples, {y_test.sum()} stressed ({y_test.mean()*100:.1f}%)")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



RARE EVENT OPTIMIZATION - Bank Early Warning System
Improving performance when stress prevalence < 10%

📊 Data Split:
   Training (2015-2022): 544 samples, 242 stressed (44.5%)
   Test (2023-2025): 166 samples, 13 stressed (7.8%)


# APPROACH 1: Cost-Sensitive XGBoost

In [4]:
print("\n" + "="*70)
print("APPROACH 1: Cost-Sensitive Learning")
print("="*70)

# Scale_pos_weight is usually (# negatives / # positives)
# For rare events, we want LOWER weight (penalize false positives more)
# Default scale_pos_weight = 3.3 (since 76% healthy / 24% stressed)

# Try different weights to penalize false positives
weight_options = [0.5, 1.0, 1.5, 2.0, 3.3]  # 3.3 is original
results_cost = []

for weight in weight_options:
    model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=weight,  # Lower = penalize false positives more
        random_state=42,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    
    model.fit(X_train_scaled, y_train)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # At threshold 0.5
    y_pred = (y_proba >= 0.5).astype(int)
    
    tp = np.sum((y_pred == 1) & (y_test == 1))
    fp = np.sum((y_pred == 1) & (y_test == 0))
    fn = np.sum((y_pred == 0) & (y_test == 1))
    
    precision = tp / (tp + fp) if (tp+fp) > 0 else 0
    recall = tp / (tp + fn) if (tp+fn) > 0 else 0
    
    results_cost.append({
        'scale_pos_weight': weight,
        'precision': precision,
        'recall': recall,
        'alerts': tp+fp,
        'correct': tp
    })

results_cost_df = pd.DataFrame(results_cost)
print("\n📊 Cost-Sensitive Results (Test: 2023-2025):")
print(results_cost_df.round(4).to_string(index=False))



APPROACH 1: Cost-Sensitive Learning

📊 Cost-Sensitive Results (Test: 2023-2025):
 scale_pos_weight  precision  recall  alerts  correct
              0.5     0.3871  0.9231      31       12
              1.0     0.3793  0.8462      29       11
              1.5     0.4138  0.9231      29       12
              2.0     0.3750  0.9231      32       12
              3.3     0.3333  1.0000      39       13


# APPROACH 2: Isolation Forest (Anomaly Detection)

In [5]:
print("\n" + "="*70)
print("APPROACH 2: Isolation Forest (Anomaly Detection)")
print("="*70)

# Train on HEALTHY banks only
healthy_mask = y_train == 0
X_healthy = X_train_scaled[healthy_mask]

iso_forest = IsolationForest(
    contamination=0.1,  # Expect ~10% anomalies
    random_state=42,
    n_estimators=100
)

iso_forest.fit(X_healthy)

# Predict on test set
anomaly_scores = iso_forest.decision_function(X_test_scaled)
anomaly_pred = (anomaly_scores < -0.1).astype(int)  # Negative score = anomaly

tp_iso = np.sum((anomaly_pred == 1) & (y_test == 1))
fp_iso = np.sum((anomaly_pred == 1) & (y_test == 0))
fn_iso = np.sum((anomaly_pred == 0) & (y_test == 1))

precision_iso = tp_iso / (tp_iso + fp_iso) if (tp_iso+fp_iso) > 0 else 0
recall_iso = tp_iso / (tp_iso + fn_iso) if (tp_iso+fn_iso) > 0 else 0

print(f"\n📊 Isolation Forest Results:")
print(f"   Precision: {precision_iso:.4f}")
print(f"   Recall: {recall_iso:.4f}")
print(f"   Alerts: {tp_iso+fp_iso}, Correct: {tp_iso}")



APPROACH 2: Isolation Forest (Anomaly Detection)

📊 Isolation Forest Results:
   Precision: 0.0455
   Recall: 0.0769
   Alerts: 22, Correct: 1


# APPROACH 3: Threshold Tuning with F-beta Score

In [6]:
print("\n" + "="*70)
print("APPROACH 3: F-beta Optimization (Favor Precision)")
print("="*70)

# Use best model from approach 1 (scale_pos_weight=1.0)
best_model = XGBClassifier(
    n_estimators=100, max_depth=6,
    scale_pos_weight=1.0,  # Best from above
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
best_model.fit(X_train_scaled, y_train)
y_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]

# Find threshold that maximizes F2 (prioritize precision less) or F0.5 (prioritize precision more)
thresholds = np.arange(0.3, 0.95, 0.05)
f_beta_results = []

beta = 0.5  # F0.5 weights precision 2x more than recall

for thresh in thresholds:
    y_pred_thresh = (y_proba_best >= thresh).astype(int)
    
    tp = np.sum((y_pred_thresh == 1) & (y_test == 1))
    fp = np.sum((y_pred_thresh == 1) & (y_test == 0))
    fn = np.sum((y_pred_thresh == 0) & (y_test == 1))
    
    precision = tp / (tp + fp) if (tp+fp) > 0 else 0
    recall = tp / (tp + fn) if (tp+fn) > 0 else 0
    
    # F-beta score
    f_beta = (1 + beta**2) * (precision * recall) / ((beta**2 * precision) + recall) if (precision+recall) > 0 else 0
    
    f_beta_results.append({
        'threshold': thresh,
        'precision': precision,
        'recall': recall,
        'f_beta': f_beta
    })

f_beta_df = pd.DataFrame(f_beta_results)
best_f_beta = f_beta_df.loc[f_beta_df['f_beta'].idxmax()]

print("\n📊 F0.5 Optimization (Preference for Precision):")
print(f_beta_df.round(4).to_string(index=False))
print(f"\n✅ Best threshold for F0.5: {best_f_beta['threshold']:.2f}")
print(f"   → Precision: {best_f_beta['precision']:.4f}")
print(f"   → Recall: {best_f_beta['recall']:.4f}")




APPROACH 3: F-beta Optimization (Favor Precision)

📊 F0.5 Optimization (Preference for Precision):
 threshold  precision  recall  f_beta
      0.30     0.3023  1.0000  0.3514
      0.35     0.3333  1.0000  0.3846
      0.40     0.3421  1.0000  0.3939
      0.45     0.3333  0.9231  0.3822
      0.50     0.3143  0.8462  0.3595
      0.55     0.3438  0.8462  0.3901
      0.60     0.3548  0.8462  0.4015
      0.65     0.3793  0.8462  0.4264
      0.70     0.3793  0.8462  0.4264
      0.75     0.4400  0.8462  0.4867
      0.80     0.4286  0.6923  0.4639
      0.85     0.4737  0.6923  0.5056
      0.90     0.4706  0.6154  0.4938

✅ Best threshold for F0.5: 0.85
   → Precision: 0.4737
   → Recall: 0.6923


# FINAL COMPARISON

In [7]:
print("\n" + "="*70)
print("FINAL MODEL COMPARISON (2023-2025 Rare Period)")
print("="*70)

comparison = pd.DataFrame({
    'Model': ['Original XGB (weight=3.3, thresh=0.5)',
              'Cost-Sensitive (weight=1.0, thresh=0.5)',
              'Isolation Forest',
              'Cost-Sensitive + F0.5 (thresh=0.80)'],
    'Precision': [0.20,  # Original from your backtest (2024 was 0.20)
                  results_cost_df[results_cost_df['scale_pos_weight']==1.0]['precision'].values[0],
                  precision_iso,
                  best_f_beta['precision']],
    'Recall': [0.75,  # Original from your backtest (2024 was 0.75)
               results_cost_df[results_cost_df['scale_pos_weight']==1.0]['recall'].values[0],
               recall_iso,
               best_f_beta['recall']]
})

print(comparison.round(4).to_string(index=False))




FINAL MODEL COMPARISON (2023-2025 Rare Period)
                                  Model  Precision  Recall
  Original XGB (weight=3.3, thresh=0.5)     0.2000  0.7500
Cost-Sensitive (weight=1.0, thresh=0.5)     0.3793  0.8462
                       Isolation Forest     0.0455  0.0769
    Cost-Sensitive + F0.5 (thresh=0.80)     0.4737  0.6923


# SAVE THE OPTIMIZED MODEL

In [8]:


print("\n" + "="*70)
print("SAVING OPTIMIZED MODEL")
print("="*70)

# Save the best cost-sensitive model
optimized_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=1.0,  # Optimized for rare events
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
optimized_model.fit(X_train_scaled, y_train)

import joblib
joblib.dump(optimized_model, r"D:\Bank_EWS_Project\models\bank_stress_rare_event.pkl")
joblib.dump(scaler, r"D:\Bank_EWS_Project\models\scaler_rare_event.pkl")

print("✅ Optimized model saved to: D:\\Bank_EWS_Project\\models\\bank_stress_rare_event.pkl")

print("\n✅ Rare event optimization complete!")


SAVING OPTIMIZED MODEL
✅ Optimized model saved to: D:\Bank_EWS_Project\models\bank_stress_rare_event.pkl

✅ Rare event optimization complete!
